In [2]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,path,datapointID,subtaskID,answer
0,./recordings/1.wav,1,1,7
1,./recordings/2.wav,2,1,6
2,./recordings/3.wav,3,1,7
3,./recordings/4.wav,4,1,7
4,./recordings/5.wav,5,1,7


In [3]:
import librosa

def get_feats(path):
    y, sr = librosa.load(path)
    mfcc = librosa.feature.mfcc(y=y, sr=sr)
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    power = librosa.feature.chroma_stft(y=y, sr=sr)
    contr = librosa.feature.spectral_contrast(S=np.abs(librosa.stft(y)), sr=sr)
    rms = librosa.feature.rms(S=librosa.magphase(librosa.stft(y, window=np.ones, center=False))[0])
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    return [mfcc.mean(), mfcc.std(), mfcc.min(), mfcc.max(), np.quantile(mfcc, 0.25), np.quantile(mfcc, 0.75), np.median(mfcc),
            mel.mean(), mel.std(), mel.min(), mel.max(), np.quantile(mel, 0.25), np.quantile(mel, 0.75), np.median(mel),
            contr.mean(), contr.std(), contr.min(), contr.max(), np.quantile(contr, 0.25), np.quantile(contr, 0.75), np.median(contr),
            rms.mean(), rms.std(), rms.min(), rms.max(), np.quantile(rms, 0.25), np.quantile(rms, 0.75), np.median(rms),
            bw.mean(), bw.std(), bw.min(), bw.max(), np.quantile(bw, 0.25), np.quantile(bw, 0.75), np.median(bw),
            power.mean(), power.std(), power.min(), power.max(), np.quantile(power, 0.25), np.quantile(power, 0.75), np.median(power)] + list(np.mean(mfcc, axis=1))

len(get_feats(df_train.path[0]))

62

In [4]:
from tqdm import tqdm

X = []
y = df_train.answer

for path in tqdm(df_train.path):
    feats = get_feats(path)
    X.append(feats)

X_final = []
for path in tqdm(df_test.path):
    feats = get_feats(path)
    X_final.append(feats)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_final = scaler.transform(X_final)

X = pd.DataFrame(X)
X_final = pd.DataFrame(X_final)
X.head()

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:07<00:00, 27.66it/s]


,0,1,2,3,4,5,6,7,8,9,...,52,53,54,55,56,57,58,59,60,61
0,-0.570878,-0.398603,0.356037,-0.064599,-0.291155,-2.471128,-1.566176,-0.640287,-0.469032,-0.464412,...,-0.423847,-1.040640,-1.864294,-1.847617,-1.648682,-1.562147,-1.556639,-1.602959,-1.677606,-1.755461
1,-0.349774,-0.325847,0.509382,0.711292,-0.773649,1.053049,0.735344,1.071725,0.687002,1.484625,...,-0.845485,-0.715429,-0.011499,0.749303,1.141261,1.418454,1.681093,1.920655,2.094815,2.157182
2,0.764220,0.839066,-0.951703,-0.409008,0.953575,0.511577,-0.360752,-0.743152,-0.671625,-0.510485,...,0.727904,0.663567,0.299991,-0.135198,-0.321289,-0.401437,-0.434588,-0.426662,-0.375864,-0.278677
3,0.475171,-1.014179,0.937026,0.306548,0.838086,-2.307095,-1.148296,0.282701,0.855290,-0.494003,...,0.434345,-0.355212,-1.730459,-2.206644,-2.156893,-2.074134,-2.002998,-1.909142,-1.771640,-1.578783
4,-0.115173,-0.175448,-0.169726,-0.606200,-0.186909,-0.649236,-0.831732,0.016886,-0.244334,-0.467288,...,0.036040,0.535022,1.175103,1.146222,0.898306,0.674714,0.466353,0.259427,0.048518,-0.142781


In [5]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=10, scoring='accuracy').mean()

np.float64(0.8875)

In [6]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [7]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.datapointID,
    "answer": y_preds
})

submission

,subtaskID,datapointID,answer
0,1,1,7
1,1,2,7
2,1,3,6
3,1,4,6
4,1,5,7
...,...,...,...
195,1,196,6
196,1,197,6
197,1,198,6
198,1,199,6


In [8]:
submission.to_csv("submission.csv", index=False)